# ROP Model Metric Evaluation & Prediction Explorer
Loads saved out-of-fold probability arrays (`.npy`) alongside `imgs_df`, `occ_df`, and `sev_df` to inspect predictions and compute image-level and patient-level evaluation metrics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

def load_df(stem):
    for ext, reader in [
        (".csv", lambda p: pd.read_csv(p, index_col=0)),
        (".parquet", pd.read_parquet),
        (".pkl", pd.read_pickle),
    ]:
        path = Path(f"{stem}{ext}")
        if path.exists():
            return reader(path)
    return None

In [ ]:
# 1. Load saved .npy arrays and DataFrames (imgs_df, occ_df, sev_df)
npy_files = sorted(Path(".").glob("*.npy"))
print(f"Found {len(npy_files)} .npy file(s): {[f.name for f in npy_files]}")
arrays = {f.stem: np.load(f) for f in npy_files}

dfs = {}
for name in ["imgs_df", "occ_df", "sev_df"]:
    df = load_df(name)
    if df is not None:
        dfs[name] = df
        print("=" * 60)
        print(f"{name} | Shape: {df.shape}")
        display(df.head())
    else:
        print(f"{name} not found in current folder (checked .csv, .parquet, .pkl)")

In [ ]:
# 2. Preview raw .npy prediction arrays
for name, arr in arrays.items():
    print("=" * 60)
    print(f"File: {name}.npy | Shape: {arr.shape} | Dtype: {arr.dtype}")
    print(f"Min: {arr.min():.4f} | Max: {arr.max():.4f} | NaNs: {np.isnan(arr).sum()}")
    if arr.ndim == 2:
        df_preview = pd.DataFrame(arr, columns=[f"class_{i}" for i in range(arr.shape[1])])
        df_preview["pred_class"] = arr.argmax(axis=1)
        display(df_preview.head(10))

In [ ]:
# 3. Metric evaluation helper (aligns oof rows via task DataFrame index)
def evaluate_task(df, oof, class_names, task_label="Task"):
    print("=" * 60)
    print(f"EVALUATION: {task_label}")
    y_true = df["y"].to_numpy()
    probs = oof[df.index.to_numpy()]
    y_pred = probs.argmax(axis=1)

    if "fold" in df.columns:
        folds = df["fold"].to_numpy()
        accs = [accuracy_score(y_true[folds == k], y_pred[folds == k]) for k in sorted(df["fold"].unique())]
        f1s  = [f1_score(y_true[folds == k], y_pred[folds == k], average="macro") for k in sorted(df["fold"].unique())]
        print(f"Fold Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f} | Fold Macro-F1: {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")

    n_classes = len(class_names)
    if n_classes == 2:
        auc = roc_auc_score(y_true, probs[:, 1])
        pr_auc = average_precision_score(y_true, probs[:, 1])
        kappa = cohen_kappa_score(y_true, y_pred)
        print(f"Pooled ROC-AUC: {auc:.3f} | PR-AUC: {pr_auc:.3f} | Cohen's Kappa: {kappa:.3f}")
    else:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        kappa = cohen_kappa_score(y_true, y_pred)
        print(f"Pooled Macro OVR ROC-AUC: {auc:.3f} | Unweighted Kappa: {kappa:.3f} | Quadratic Weighted Kappa: {qwk:.3f}")

    labels = list(range(n_classes))
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=[f"actual {c}" for c in class_names],
        columns=[f"pred {c}" for c in class_names],
    )
    display(cm)
    print(classification_report(y_true, y_pred, labels=labels, target_names=class_names, digits=3, zero_division=0))

# Run evaluation for available task DataFrames & OOF arrays
if "occ_df" in dfs and "oof_occ" in arrays:
    evaluate_task(dfs["occ_df"], arrays["oof_occ"], ["Normal", "ROP"], task_label="ROP Occurrence")

if "sev_df" in dfs and "oof_sev" in arrays:
    evaluate_task(dfs["sev_df"], arrays["oof_sev"], ["Mild ROP", "Moderate ROP", "Severe ROP"], task_label="ROP Severity")